# 03 — Feature Engineering & Preprocessing

**What we engineer (and why):**

| Feature | From | Rationale |
|---|---|---|
| `Hour` | `Time` | Hour-of-day (0–23). Fraud rate spikes at night. Raw `Time` (seconds since collection start) is an artifact of the 2-day window and is dropped. |
| `Amount_log` | `Amount` | `log1p` tames extreme right skew so the linear model isn't dominated by outliers. Raw `Amount` is kept in the frame for business-cost analysis but excluded from model features. |

**What we leave alone:** `V1–V28` are already PCA outputs — centered, orthogonal, informative.
Re-transforming them adds nothing.

**Scaling.** `StandardScaler` lives *inside* the model pipeline, so it is fit only on training folds
— never on validation/test (leakage-safe by construction). Trees don't need scaling but it is
harmless, and one uniform pipeline keeps training and serving identical.

**Leakage checklist applied here:**
1. Duplicates dropped **before** splitting (no identical row in train and test).
2. Feature engineering is row-wise only (no statistics learned from the full dataset).
3. Scaler and resamplers are pipeline steps → fit on training data only.
4. Threshold tuned on **validation**, test set touched exactly once.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import Image, display
from src.utils import load_config, resolve_path

config = load_config()
pd.set_option("display.max_columns", 40)

In [ ]:
from src.data_loader import load_raw_data
from src.preprocessing import drop_duplicates, make_splits
from src.feature_engineering import engineer_features, get_feature_columns

df = drop_duplicates(load_raw_data(config))
df_fe = engineer_features(df, config)
feature_cols = get_feature_columns(df_fe, config)
print(f"{len(feature_cols)} model features:")
print(feature_cols)

Sanity check — fraud rate by engineered `Hour` (the night-time spike the feature captures):

In [ ]:
hourly = df_fe.groupby("Hour")["Class"].agg(n="size", frauds="sum", rate="mean")
hourly["rate_pct"] = (100 * hourly.pop("rate")).round(3)
hourly.T

## Stratified three-way split

- **Test (20%)** — untouched until final evaluation; simulates future unseen transactions.
- **Validation (16%)** — model selection + threshold tuning.
- **Train (64%)** — model fitting.

`stratify=y` keeps the 0.17% fraud rate identical in all three — with 492 frauds total, an
unstratified split could easily produce a test set with too few frauds to evaluate on.

In [ ]:
splits = make_splits(df_fe, feature_cols, config)
pd.DataFrame({
    "rows": [len(splits.y_train), len(splits.y_val), len(splits.y_test)],
    "frauds": [int(s.sum()) for s in (splits.y_train, splits.y_val, splits.y_test)],
    "fraud_%": [round(100 * s.mean(), 4) for s in (splits.y_train, splits.y_val, splits.y_test)],
}, index=["train", "val", "test"])